## 1. Helpers for GPT model

In [ ]:
%%capture
! pip install openai
! pip install langchain
! pip install langchain-community
! pip install pypdf
! pip install tiktoken

In [ ]:
# function for GPT text generation

def new_message_object(role, content):
  return {"role": role, "content": content}


def run_completion_gpt(messages, model = "gpt-4o", temperature=0, response_format = None):
  if response_format:
    response = client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        temperature=temperature,
        response_format=response_format
        )
  else:
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
        )


  output = response.choices[0].message.content

  return output

In [ ]:
import os

# Set your OpenAI API key as an environment variable before running this notebook.
# Recommended: export it in your shell (or use a secrets manager) instead of
# hardcoding it here, e.g.:  export OPENAI_API_KEY="your-api-key"
# The OpenAI client created below reads OPENAI_API_KEY from the environment automatically.

# os.environ['OPENAI_API_KEY'] = 'your-api-key-here'

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

import re
import json
from tqdm import tqdm
import ast

In [ ]:
from openai import OpenAI
client = OpenAI()

## 2. Load papers in pdf and transfer to text

In [ ]:
import os
import json
from langchain.document_loaders import PyPDFLoader

def extract_pdfs_to_json(pdf_folder_path, json_path):
    """
    Extracts text from all PDFs in the given folder and saves them to a JSON file.
    If the JSON file exists, it is replaced with an empty JSON before processing.

    Args:
        pdf_folder_path (str): Path to the folder containing PDF files.
        json_path (str): Path to the JSON file to save the extracted texts.
    """
    # If the JSON file exists, replace it with an empty JSON
    with open(json_path, 'w') as f:
        json.dump({}, f)

    all_pdfs = {}
    for file_name in os.listdir(pdf_folder_path):
        if file_name.lower().endswith('.pdf'):
            pdf_idx = file_name.rsplit('.pdf', 1)[0]
            loader = PyPDFLoader(os.path.join(pdf_folder_path, file_name))
            document = loader.load()
            pdf_text = ''
            for page in document:
                pdf_text += page.page_content
            all_pdfs[pdf_idx] = pdf_text
    with open(json_path, 'w') as fp:
        json.dump(all_pdfs, fp)

In [ ]:
pdf_path = 'data/pdf_to_review'
pdf_text_json_path = 'data/pdf_text.json'
extract_pdfs_to_json(pdf_path, pdf_text_json_path)

## 3. Scope review - single step


In [ ]:
import re
import pandas as pd

def remove_parentheses_and_convert_to_list(s):
    """
    Remove all ( ... ) including content from a string,
    then split by commas to form a clean list.
    """
    # Remove everything between parentheses (including parentheses):
    # For instance, "TT (regional analgesia, multimodal)" becomes "TT ".
    s = re.sub(r"\(.*?\)", "", s)

    # Remove the surrounding square brackets, if any
    s = s.strip("[]")

    # Split by comma
    items = s.split(",")

    # Clean up extra whitespace and ignore empty strings
    cleaned_items = [item.strip() for item in items if item.strip()]

    return cleaned_items


def clean_dataframe_column(df, col_name_list):
    """
    Apply the remove_parentheses_and_convert_to_list function
    to a specific column in your DataFrame.
    """
    for col_name in col_name_list:
      df[col_name] = df[col_name].apply(remove_parentheses_and_convert_to_list)
    return df


In [ ]:
# load all pdf texts
import json
import pandas as pd

pdf_text_json_path = 'data/pdf_text.json'
with open(pdf_text_json_path) as f:
    all_pdfs = json.load(f)

df_all_pdfs = pd.DataFrame(all_pdfs.items(), columns=['article_id', 'pdf_text']).sort_values(by='article_id')
df_all_pdfs['pdf_text_length'] = df_all_pdfs['pdf_text'].apply(len)

In [ ]:
df_all_pdfs[['article_id', 'pdf_text_length']]

In [ ]:
analisys_prompt_1_1 = """
<role>You are a research assistant with expertise in pain and sociology.</role>

<task>
You will be given articles about pain or pain treatment to analyze. \
For each article, your task is to read the article carefully and summarize the main research goal and outcomes. \
Each article discusses (a) one or more pain and/or pain treatment outcomes, \
(b) one or more social dimensions (e.g., sex, race, age, or social class) that could be associated with the pain / pain treatment outcome, and \
(c) one or more mechanisms which could explain any observed social group differences or disparities in pain / pain treatment.  \
The pain and/or pain treatment outcome(s) in (a) are considered “dependent variables”, \
and the social dimensions(s) in (b) are considered “independent variables.”
</task>

<instruction>
1. Read the article carefully, especially the TITLE and ABSTRACT.
2. Based on the abstract or introduction, identify and extract the research objective/goal/aim/purpose using the article’s original wording. If the article does not explicitly state the research aim, summarize the research goal in your own words.
3. Based on the research goal, identify and summarize each distinct aspect or dimension of the goal in one sentence. Note that there may be one or multiple aspects to the research goal. But one aspect of research goal can include multiple research findings.
4. Distinguish the core research content and findings from the introductory or background material.
5. Based on the aspects of the research goal you identified and core research content, determine the main research findings and summarize each main finding in one sentence. Note that each research finding should only pertain to one aspect of research objective.
6. For each research finding, answer the following four questions:
- 6.a). What is the broad type of pain or pain treatment outcome (e.g., acute experimental pain, cancer pain, a specific kind of chronic pain, post-surgical pain, etc.)?
- 6.b).	What is the specific aspect of pain or pain treatment being studied (e.g., prevalence, intensity, pain threshold, prescription of a drug, etc.)?
- 6.c).	What social dimension(s) are associated with this type of pain / pain treatment (e.g., sex, race, age, social class)?
- 6.d).	For each social dimension, what mechanism(s) (if any) are proposed to explain its association (if any) with the pain or pain treatment outcome?  This association may be referred to as a “difference,” “disparity,” or “inequity,” among other terms.
7. Organize your answers following the format defined below in <output_format>.
</instruction>

<output_format>
Save your output in JSON with the following format and organize it by research goal, each aspect of the research goal and each research finding:
{
  "research_goal": "research goal summary in one sentence using the article’s original wording if available or write your own one-sentence summary if no goal is stated",
  "research_goal_aspect":
  {
    "research_goal_aspect_1": "research goal aspect summary in one sentence using the article's original wording if available or write your own one-sentence summary if the aspect of the goal is not clear",
    "research_goal_aspect_2": "research goal aspect summary in one sentence using the article's original wording if available or write your own one-sentence summary if the aspect of the goal is not clear",
    ...
  },
  "research_finding":
  {
    "research_finding_1": "research finding summary in one sentence which should be specific and only focused on one objective with dependent and independent variables clearly specified",
    "research_finding_2": "research finding summary in one sentence which should be specific and only focused on one objective with dependent and independent variables clearly specified",
    ...
  },
  "research_finding_details": {
    "research_finding_1": {
      "6.a": "Broad type: ...",
      "6.b": "Specific aspect: ...",
      "6.c": "Social dimension: ...",
      "6.d": "Mechanisms: ..."
    },
    "research_finding_2": {
      "6.a": "Broad type: ...",
      "6.b": "Specific aspect: ...",
      "6.c": "Social dimension: ...",
      "6.d": "Mechanisms: ..."
    },
    ...
  }
}

</output_format>

<article_to_analyze>
{pdf_text}
</article_to_analyze>

Your research outcomes in JSON:
"""

In [ ]:
analisys_prompt_1_2 = """
<role>You are a research assistant with expertise in pain and sociology.</role>

<task>
You will be given the research goal and findings which were identified and extracted from an article about pain or pain treatment to analyze. \
For each research finding, your task is to read the research finding carefully and categorize the pain or treatment type, aspect type, social dimensions type, mechanism type using the answering options given below. \
Each research finding discusses (a) one pain and/or pain treatment outcomes, \
(b) one social dimension (e.g., sex, race, age, or social class) that could be associated with the pain / pain treatment outcome, and \
(c) one mechanism which could explain any observed social group differences or disparities in pain / pain treatment.  \
</task>

<instruction>
1. Read the research goal and research findings.
2. For each research finding, answer the following four questions:
- 2.a). What is the broad type of pain or pain treatment outcome (e.g., acute experimental pain, cancer pain, a specific kind of chronic pain, post-surgical pain, etc.)?
- 2.b). What is the specific aspect of pain or pain treatment being studied (e.g., prevalence, intensity, pain threshold, prescription of a drug, etc.)?
- 2.c). What social dimension(s) are associated with this type of pain / pain treatment (e.g., sex, race, age, social class)?
- 2.d). For each social dimension, what mechanism(s) (if any) are proposed to explain its association (if any) with the pain or pain treatment outcome?  This association may be referred to as a “difference,” “disparity,” or “inequity,” among other terms.
3. For each of the four items identified for each research finding (described just above), select the most relevant pre-specified category or categories:
- 3.a). For the pain or pain treatment identified in 2.a), select the most relevant category or categories from <pain_or_treatment_types>; refer <notes_for_pain_or_treatment_types> for important notes
- 3.b). For the specific aspect of pain or pain treatment identified in 2.b), select the most relevant category or categories from <aspect_types>
- 3.c). For the social dimension(s) identified in 2.c), select the most relevant category or categories from <social_dimensions_types>
- 3.d). For the mechanism(s) proposed to explain social group differences or disparities in pain or pain treatment identified in 2.d), select the most relevant category or categories from <mechanism_types>
4. Organize your answers following the format defined below in <output_format>.
</instruction>

<pain_or_treatment_types>
- AE: Acute experimental pain (e.g., laboratory studies with humans or with animals like mice or rats examining responses to induced pain, including pain tolerance and pain thresholds).
- AS: Specific acute pain (non-experimental, with a specific type or location, e.g., acute back pain).
- AG: General acute pain (no specific type mentioned, e.g., simply “acute pain”).
- CA: Cancer pain (specific or general, e.g., lung cancer pain, general cancer-related pain or disease-specific pain).
- CS: Chronic specific pain (refers to chronic pain associated with specific conditions, such as fibromyalgia, chronic low back pain, or irritable bowel syndrome (IBS), where the pain is long-term and linked to a known underlying medical condition).
- CG: Chronic general pain (refers to long-term, persistent pain without a specific type identified, often broadly described as "chronic pain" without a clear underlying condition or precise localization).
- PS: Specific pain (refers to pain that is specific to a particular body part or condition but is not explicitly classified as chronic or acute. For example, conditions without clear classification as chronic or acute, such as low back pain, temporomandibular disorders (TMD), migraine).
- PG: General pain (refers to unspecified pain without detailed characterization, often described simply as "pain" without specifying whether it is chronic, acute, or related to a specific type.).
- SU: Post-surgical pain (e.g., postoperative pain).
- TM: Pain treatment (refers to methods for diagnosing, managing, and coping strategies, including surgery, medication, and other therapeutic approaches).
- TR: Treatment response (refers to the outcomes or effectiveness of pain management strategies, including the degree of symptom relief, functional improvement, and overall impact of treatments such as medication, physical therapy, or surgical interventions. In some lab articles, if they mention different reactions of mice for some stimuli, TR should also be included.)
</pain_or_treatment_types>

<notes_for_pain_or_treatment_types>
a. Always base your decision on the stated research goal. For example, you can only select TM or TR or both if the research goal mentions they aim to examine social group differences in pain management, diagnosis, coping strategies, medications, interventions, or surgery. Otherwise, if they only mention this in the research finding, do not select TM or TR.
b. Never infer a code based on disease population or sample context if it is not named in the aim or finding sentence. Do not overthink or guess about chronicity or acuteness; use only what is stated in the research goal or findings.
c. The rationale for selecting a category from </pain_or_treatment_types> is to determine whether the article discusses social group differences in this specific pain type. The type should only be chosen if this criterion is explicitly met. For example, if research finding states, "Specific hormone regimens (e.g., estradiol valerate/dienogest, extended-cycle vaginal ring contraception) can reduce headache and aura frequency in women with MRM and MwA," this is not a comparison, as it only reports benefits in women without comparing outcomes between women and men. For findings like this, do not code a pain type related to social group differences.
</notes_for_pain_or_treatment_types>

<aspect_types>
- INC: pain incidence, only code this if the paper includes this key word.
- PR: pain prevalence, only code this if the paper includes this key word.
- TH: pain thresholds or pain tolerance.
- INT: pain intensity or pain severity
- DU: pain duration
- FN: pain-related function (e.g., ability to walk; disability; return to work)
- RX: pain medication prescription (e.g., yes/no, or type of Rx). Specifically, if the paper discusses "opioid" or "medication" or "drug" or "medicine", it might be related to RX. For example, in the article related to pain treatment, if it mentions that some social groups are more/less likely to be prescribed certain drugs than are other groups, code RX.
- DO: the dosage of Rx. Dosage means how much of the drug a person gets, for example, do some social groups get prescribed 50 mg or morphine, or 200 mg of morphine? Perhaps some groups get higher dosages than others. That is, if the paper discusses specific dose/dosage of medication, code DO.
- SP: surgery or other procedure (e.g., knee replacement, hip replacement, abdominal surgery, steroid injections).
- TT: other treatment types (e.g., herbal remedies, physical therapy, music therapy, complementary and alternative treatments).
- OT: other, for aspects that do not fall into any of the above categories (e.g., pain coping, pain beliefs, access to pain care). Pain-related mental health (e.g., depression) and pain assessment and diagnosis should be coded as OT.
</aspect_types>

<social_dimensions_types>
- AGE: Age (e.g., comparisons between younger adults and older adults or changes in pain with increasing age).
- SEX: Gender or biological sex (or both).
- RAC: Race and/or ethnicity (e.g., comparisons between White and Black individuals, or between cultural or ethnic groups like Miao vs. others).
- SES: Socioeconomic status (e.g., comparisons by education, income, occupation, or wealth).
- INS: Health insurance status (e.g., insured vs. uninsured).
- WCG: Within-country geographic comparison (e.g., rural vs. urban, or neighborhood-level comparisons).
- CCG: Cross-country or large geographic comparisons (e.g., European countries vs. the U.S. or continent-level comparisons).
- OTH: Other social dimensions not listed above.
</social_dimensions_types>

<mechanism_types>
- BG: Biological genetic mechanisms (e.g., genes). Common in papers on gender/sex differences.
- BH: Biological hormonal mechanisms (e.g., testosterone, estrogen). Common in papers on gender/sex/sexuality differences.
- BN: Biological neurophysiology (e.g., brain differences, μ-opioid receptors). Common in papers on gender/sex or age differences.
- BO: Other or unspecified biological factors (e.g., comorbidity, inflammation system, immune system, antibody responses). Common in papers on gender/sex or racial/ethnic differences
- SC: Sociocultural factors (e.g., lifestyles, reporting differences). Common in papers on race/ethnicity, SES, age, sex/gender differences. Examples: lifestyle differences (e.g., Miao ethnicity has distinct diets compared to other groups); reporting differences (e.g., women report pain more often than men due to social norms).
- PS: Psychosocial factors (e.g., stress, mental health). Common in papers on race/ethnicity, SES, sex/gender differences.  Example: studies show higher stress or depression among Black individuals compared to White individuals.
- PF: Provider factors (e.g., treatment decisions, bias).  Common in papers on gender/sex/sexuality or race/ethnicity differences.  Example: physician bias leads to different opioid prescription rates for non-White vs. White patients.
- HS, health system factors (e.g., lack of health insurance, lack of affordable or accessible healthcare, or lack of translators in hospitals). Common in papers on rural/urban, cross-country, or racial/ethnic differences.
- EN, other macro-level environmental (sociostructural) factors (e.g., geographic contexts, health policies, local opioid availability, racism, neighborhood factors, or other structural social factors other than health systems factors).
- OT, other (explain in the parenthesis) (Anything that does not fall into any of the categories above)
- NA, if no mechanisms were discussed in the paper.
</mechanism_types>

<output_format>
Save your output in JSON with the following format and organize it by each given research finding :
{
  "research_finding_1":
  {
    "pain_or_treatment_type": "select one value from <pain_or_treatment_types>",
    "aspect_type": "select one value from <aspect_types>",
    "social_dimensions_type": "select one value from <social_dimensions_types>",
    "mechanism_type": "select one value from <mechanism_types>",
  },
  "research_finding_2":
  {
    "pain_or_treatment_type": "select one value from <pain_or_treatment_types>",
    "aspect_type": "select one value from <aspect_types>",
    "social_dimensions_type": "select one value from <social_dimensions_types>",
    "mechanism_type": "select one value from <mechanism_types>",
  },
  ...
}
</output_format>

<research_goal_and_finding_to_analyze>
{research_goal_and_finding_to_analyze}
</research_goal_and_finding_to_analyze>

Your research outcomes in JSON:
"""

In [ ]:
analisys_prompt_2 = """
<role>You are a research assistant with expertise in pain and sociology.</role>

<task>
Given the article text, your task is to read the article about pain / pain treatment carefully \
and determine what type of review the article is and whether it reported statistically significant findings.
</task>

<instruction>
1. Read the article carefully, especially the title, abstract, method and finding sections.
2. Separate the core research contents and outcomes / findings from the background introduction.
3. Summarize the main research contents and findings.
4. Given your summary and your knowledge of the article, determine the following components of the article:
- 4.a). Identify the review type of this article. Pick one value from: SRQuan, SRQual, MA, QA, and strictly follow the instruction provided in <how_to_define_review_types>
- 4.b). If the article type is SRQuan or MA, please indicate whether the findings section reports statistically significant differences in pain or pain treatment across social groups. Determine statistical significance, and strictly follow the instructions provided in <how_to_define_statistical_significance>
- 4.c). What are the social dimension(s) associated with this pain or pain treatment outcome? Identify the key social groups being compared in the article’s findings section, and strictly follow the instructions provided in <how_to_define_key_social_groups>
- 4.d). If the article type is SRQuan or MA, does the Findings section indicate that the lower-status group have worse pain or pain treatment outcomes? Determine lower-status group significance for the social groups identified in 4.c) , and strictly follow the instructions provided in <how_to_define_lower_status_group_significance>
- 4.e). If the article type is QA or SRQual, did the authors assume that lower-status groups have worse pain or pain treatment outcomes, or test whether lower-status groups have worse pain or pain treatment outcomes? Distinguish between these options, selecting one value from <qualitative_assumption>
5. Summarize the classifications and organize them as follows:
- 5.a).	Review Type: (SRQuan, SRQual, MA, QA)
- 5.b).	Statistical Significance: (YES, NO, MIXED, NA, QA)
- 5.c).	Lower-Status Group Does Worse: (YESWORSE, MIXED, MIXEDOPP, MIXEDBETTER, BETTER, NODIFF, QA)
- 5.d).	Qualitative assumption: (YES, NO, OPP, UNC, QT)
6. Organize your answer following the format defined below in <output_format>.
</instruction>

<how_to_define_review_types>
1. If the article demonstrates the characteristics of a systematic review (e.g., specifying what databases, search terms, and inclusion criteria were used), classify it as a systematic review, even if the authors do not explicitly label it as such. That is, we code based on what the articles actually did, rather than what the authors say or didn’t say they did.
2. When coding this question, do not rely on the formal labels (e.g., scoping review, systematic review, narrative review) provided by the authors. Instead, strictly follow the instructions below to determine the review type. Avoid overthinking or making assumptions.
3. Determine Review Type by answering the following questions:
  3-1. Does the paper report using a systematic method to search the literature (e.g., searching databases like PubMed, SCOPUS, or Google Scholar with specified search terms)?  If NO, it is qualitative review and categorize it as "QA". If YES, proceed to 3-2.
  3-2. Does the paper pool data together and conduct original statistical analyses? If YES, it is meta-analysis and categorize it as "MA". If NO, proceed to 3-3.
  3-3. Does the paper report or discuss statistical differences (e.g., "statistical," or "p-value") between key independent variables (IVs) (i.e., social groups), even if these differences (or lack thereof) come from other studies? If YES, it is systematic statistical review and categorize it as "SRQuan". If NO, it is systematic qualitative review and categorize it into "SRQual". If the paper only discusses statistical findings from other studies (without conducting its own meta-analysis), classify it as SRQuan, not MA.
</how_to_define_review_types>

<how_to_define_statistical_significance>
-	If Review Type is QA or SRQual, categorize as QA.
-	If Review Type is MA or SRQuan, select code from <statistical_significance>.
-	Do not code QA if the Review Type is MA or SRQuan.
-	Important: If the study reports no statistically significant differences, classify Quantitative Significance as NO, even if the study is a meta-analysis (MA) or systematic quantitative review (SR Quan).
</how_to_define_statistical_significance>

<statistical_significance>
-	YES: A group consistently and significantly has worse pain outcomes.
-	NO: No significant differences are reported (p-value > 0.05).
-	MIXED: Some findings show significance, others do not.
-	NA: No statistical analysis performed.
</statistical_significance>

<how_to_define_key_social_groups>
The following categories should be used.
- AGE: Examines age-related differences in outcomes.
- SEX: Examines gender or sex differences in outcomes.
- RAC: Examines differences in outcomes by race and/or ethnicity.
- SES: Examines differences in outcomes by socioeconomic status, e.g., by education, income, or wealth.
- INS: Examines differences in outcomes by health insurance status (e.g., insured vs. uninsured).
- WCG: Examines differences in outcomes by within-country geographic units (e.g., rural vs. urban).
- CCG: Examines differences in outcomes across countries or larger (e.g., continental) geographic units.
- OTH: Any other relevant social dimension not already specified above.
</how_to_define_key_social_groups>

<how_to_define_lower_status_group_significance>
Determine whether Lower-Status Group Does Worse:
1. Compare findings between lower-status groups (i.e., socially disadvantaged groups like women/females, racial/ethnic minorities, older adults, people with less education or income, people from developing countries, etc.) and higher-status groups (e.g., socially advantaged groups like men/males, whites, younger adults, people with more education or income, people from highly developed countries, etc).
2. If multiple social groups are analyzed, assess the patterns across all groups rather than focusing on a single group.
3. Select code from <lower_status_significance>
4. Note that this component only applies for articles coded as “SRQuan” or “MA” for Review Type.
</how_to_define_lower_status_group_significance>

<lower_status_significance>
- YESWORSE: The lower-status group consistently has worse pain or pain treatment outcomes than the higher-status group.
- MIXED: Sometimes the lower-status group has worse outcomes than the higher-status group, but sometimes there is no significant difference.
- MIXEDOPP: Sometimes the lower-status group has worse outcomes, sometimes better outcomes, and sometimes no difference. Use MIXEDOPP when the study finds both worse and better outcomes for the lower-status group, depending on the condition examined. Example: African Americans have lower prevalence of tension-type headaches (better outcome) but higher chronic migraine burden, greater pain severity, and worse treatment access (worse outcomes).
- MIXEDBETTER: Sometimes the lower-status group has better outcomes than the higher-status group, but sometimes there is no significant difference.
- BETTER: The lower-status group consistently has better pain outcomes than the higher-status group.
- NODIFF: There is no significant difference between the lower-status and higher-status groups.
- QA: Use only if Statistical  Significance is classified as QA.
</lower_status_significance>

<qualitative_assumption>
- No: the article didn’t presume that there are differences or disparities in pain or pain treatment.
- YES: the article presumed that lower-status group(s) have worse pain/pain treatment outcomes than higher-status groups.
- OPP: the article presumed that lower-status group(s) have BETTER pain/pain treatment outcomes than higher-status groups.
- UNC: unclear if the article presumed a difference or disparity.
- QT: not applicable because the Review Type is “MA” or “SRQuan”.
</qualitative_assumption>

<output_format>
Save your output in JSON with the following format:
{
  "review_type": "select one value from SRQuan, SRQual, MA, QA",
  "statistical_significance": "select one value from YES, NO, MIXED, NA, QA",
  "lower_status_significance": "select one value from YESWORSE, MIXED, MIXEDOPP, MIXEDBETTER, BETTER, NODIFF, QA",
  "qualitative_assumption": "select one value from YES, NO, OPP, UNC, QT",
}
</output_format>

<article_to_analyze>
{pdf_text}
</article_to_analyze>

Your research outcomes in JSON:
"""

In [ ]:
import re
from collections.abc import Iterable


def extract_string(text):
    try:
      return ast.literal_eval(text)

    except:
      if "```json" not in text:
        text = text.replace('```', '```json', 1)
      return ast.literal_eval(text.split("```json")[1].split("```")[0].strip())


def flatten_list(lst):
    """Flatten a list of lists into a single list, but do not flatten strings."""
    flat = []
    for item in lst:
        if isinstance(item, list):
            flat.extend(flatten_list(item))
        else:
            flat.append(item)
    return flat


def dedup_and_flatten_lists_in_df(df):
    """
    For all columns in the DataFrame:
    - If a cell is a list, deduplicate it.
    - If a cell is a list of lists, flatten it and deduplicate.
    """
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, list)).any():
            def process_cell(cell):
                if isinstance(cell, list):
                    flat = flatten_list(cell)
                    return list(dict.fromkeys(flat))  # dedup and preserve order
                return cell
            df[col] = df[col].apply(process_cell)
    return df


def clean_NA_in_list(df):
    """
    Processes all columns in a DataFrame that contain lists.
    If a list contains only 'NA', it is left unchanged.
    If a list contains more than one item and includes 'NA', 'NA' is removed.
    """
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, list)).all():
            df[col] = df[col].apply(
                lambda lst: lst if lst == ["NA"]
                else [item for item in lst if item != "NA"]
            )
    return df


def merge_dicts(dict_list):
    combined = {
        "research_goal": None,
        "research_goal_aspect": {},
        "research_finding": {},
        "research_finding_details": {},
        "pain_or_treatment_type": [],
        "aspect_type": [],
        "social_dimensions_type": [],
        "mechanism_type": [],
        "review_type": None,
        "statistical_significance": None,
        "lower_status_significance": None,
        "qualitative_assumption": None
    }

    type_keys = ["pain_or_treatment_type", "aspect_type", "social_dimensions_type", "mechanism_type"]

    for d in dict_list:
        for key, value in d.items():
            # Flat values
            if key in combined and not isinstance(combined[key], (dict, list)):
                combined[key] = value

            # Merge dicts
            elif key in ["research_goal_aspect", "research_finding", "research_finding_details"]:
                if isinstance(value, dict):
                    combined[key].update(value)

            # Handle dicts like {"research_finding_1": {type_keys...}}
            elif key.startswith("research_finding_") and isinstance(value, dict):
                for subkey in type_keys:
                    if subkey in value:
                        combined[subkey].append(value[subkey])

    return combined


def flatten_nested_fields(df):
    flat_rows = []

    for _, row in df.iterrows():
        max_len = max(
            len(row.get("research_goal_aspect", {})),
            len(row.get("research_finding", {})),
            len(row.get("research_finding_details", {}))
        )

        for i in range(max_len):
            flat_row = {
                "paper_idx": row["paper_idx"] if i == 0 else "",
                "research_goal": row["research_goal"] if i == 0 else "",
                "research_goal_aspect": "",
                "research_finding": "",
                "research_finding_details": ""
            }

            # Fill indexed items if available
            rga_keys = list(row.get("research_goal_aspect", {}).values())
            rf_keys = list(row.get("research_finding", {}).values())
            rfd_keys = list(row.get("research_finding_details", {}).values())

            if i < len(rga_keys):
                flat_row["research_goal_aspect"] = rga_keys[i]

            if i < len(rf_keys):
                flat_row["research_finding"] = rf_keys[i]

            if i < len(rfd_keys):
                flat_row["research_finding_details"] = rfd_keys[i]

            flat_rows.append(flat_row)

    return pd.DataFrame(flat_rows)



def clean_list_like_columns(df):
    """
    Converts list-type or stringified list-type column values in a DataFrame
    into clean strings: no brackets/quotes, items separated by ', '.
    """
    def convert(val):
        # Try converting stringified lists to actual lists
        if isinstance(val, str):
            try:
                parsed = ast.literal_eval(val)
                if isinstance(parsed, list):
                    return ', '.join(str(item) for item in parsed)
            except (ValueError, SyntaxError):
                pass  # Leave as-is if not a valid list string
        elif isinstance(val, list):
            return ', '.join(str(item) for item in val)
        return val  # Return original if not list or list-like

    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, list) or (isinstance(x, str) and x.strip().startswith('['))).any():
            df[col] = df[col].apply(convert)

    return df

In [ ]:
# df_all_pdfs = df_all_pdfs[:10]

In [ ]:
# load existing review results
import json

model_name = "gpt-4.1-2025-04-14"
temperature = 0

# model_name = "o4-mini"
# temperature = 1

pdf_scope_review_json_path = f'data/pdf_scope_review_method_2_{model_name}.json'


# with open(pdf_scope_review_json_path, 'w') as f:
#   json.dump({}, f)


all_prompts_to_run = [analisys_prompt_1_1, analisys_prompt_1_2, analisys_prompt_2]

# all_results = []

with open(pdf_scope_review_json_path, 'r') as file:
  all_results = json.load(file)

cnt = 0
research_goal_and_finding_to_analyze = None

for idx, row in df_all_pdfs.iterrows():
  pdf_idx = row['article_id']
  pdf_text = row['pdf_text']

  all_prompt_results = []
  cnt += 1

  if cnt>len(all_results):
    for prompt_idx, analisys_prompt in enumerate(all_prompts_to_run):
      if prompt_idx in [0, 2]:
        prompt = [
            new_message_object("user", analisys_prompt.replace("{pdf_text}", str(pdf_text))),
            ]
      elif prompt_idx==1:
        assert research_goal_and_finding_to_analyze!=None
        prompt = [
            new_message_object("user", analisys_prompt.replace("{research_goal_and_finding_to_analyze}", str(research_goal_and_finding_to_analyze))),
            ]
        research_goal_and_finding_to_analyze = None

      # print(prompt)
      response = run_completion_gpt(messages=prompt, model=model_name, temperature=temperature)

      result_dict = extract_string(response)
      if "research_goal" in result_dict:
        research_goal_and_finding_to_analyze = result_dict

      all_prompt_results.append(result_dict)

      print(f">>>>>> {pdf_idx}\n")
      print(cnt)
      print(f"pdf text char length: {len(pdf_text)}")
      print(response)
      print(f"<<<<<< \n")

      if prompt_idx==2:
        combined_result = merge_dicts(all_prompt_results)
        all_results.append(combined_result)
        # print(all_results)

        with open(pdf_scope_review_json_path, 'w') as fp:
          json.dump(all_results, fp)

In [ ]:
results_df = pd.DataFrame(all_results)
results_df['paper_idx'] = df_all_pdfs['article_id'].tolist()
results_df = results_df[['paper_idx'] + [col for col in results_df.columns if col != 'paper_idx']]
results_df = dedup_and_flatten_lists_in_df(results_df)
results_df = clean_NA_in_list(results_df)
results_df = clean_list_like_columns(results_df)
results_df.to_csv(pdf_scope_review_json_path.replace('.json','.csv'))

flat_df = flatten_nested_fields(results_df)
flat_df.to_csv(pdf_scope_review_json_path.replace('.json','_flatten.csv'))